# ***8. Text generation with an RNN***

In [1]:
%tensorflow_version 2.x  # this line is not required unless you are in a notebook
from keras.preprocessing import sequence
import keras
import tensorflow as tf
import os
import numpy as np

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.


In [2]:
path_to_file = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [3]:
# Read, then decode for py2 compat.
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
# length of text is the number of characters in it
print ('Length of text: {} characters'.format(len(text)))

Length of text: 1115394 characters


In [4]:
# Take a look at the first 250 characters in text
print(text[:250])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



In [8]:
vocab = sorted(set(text))
# Creating a mapping from unique characters to indices
char2idx = {u:i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

def text_to_int(text):
  return np.array([char2idx[c] for c in text])

text_as_int = text_to_int(text)

In [9]:
# lets look at how part of our text is encoded
print("Text:", text[:13])
print("Encoded:", text_to_int(text[:13]))

Text: First Citizen
Encoded: [18 47 56 57 58  1 15 47 58 47 64 43 52]


In [10]:
def int_to_text(ints):
  try:
    ints = ints.numpy()
  except:
    pass
  return ''.join(idx2char[ints])

print(int_to_text(text_as_int[:13]))

First Citizen


In [64]:
seq_length = 100  # length of sequence for a training example
examples_per_epoch = len(text)//(seq_length+1)

# Create training examples / targets
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

In [65]:
sequences = char_dataset.batch(seq_length+1, drop_remainder=True)

In [66]:
def split_input_target(chunk):  # for the example: hello
    input_text = chunk[:-1]  # hell
    target_text = chunk[1:]  # ello
    return input_text, target_text  # hell, ello

dataset = sequences.map(split_input_target)  # we use map to apply the above function to every entry

In [67]:
for x, y in dataset.take(2):
  print("\n\nEXAMPLE\n")
  print("INPUT")
  print(int_to_text(x))
  print("\nOUTPUT")
  print(int_to_text(y))



EXAMPLE

INPUT
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You

OUTPUT
irst Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You 


EXAMPLE

INPUT
are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you 

OUTPUT
re all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you k


In [68]:
BATCH_SIZE = 64
VOCAB_SIZE = len(vocab)  # vocab is number of unique characters
EMBEDDING_DIM = 256
RNN_UNITS = 1024

# Buffer size to shuffle the dataset
# (TF data is designed to work with possibly infinite sequences,
# so it doesn't attempt to shuffle the entire sequence in memory. Instead,
# it maintains a buffer in which it shuffles elements).
BUFFER_SIZE = 10000

data = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

In [69]:
def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    inputs = tf.keras.Input(batch_shape=(batch_size, None))
    x = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim)(inputs)
    x = tf.keras.layers.LSTM(rnn_units,
                             return_sequences=True,
                             stateful=True,
                             recurrent_initializer='glorot_uniform')(x)
    outputs = tf.keras.layers.Dense(vocab_size)(x)
    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model(VOCAB_SIZE, EMBEDDING_DIM, RNN_UNITS, BATCH_SIZE)
model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (64, None)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_3 (Embedding)         │ (64, None, 256)        │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (64, None, 1024)       │     5,246,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (64, None, 65)         │        66,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,330,241 (20.33 MB)

 Trainable params: 5,330,241 (20.33 MB)

 Non-trainable params: 0 (0.00 B)

In [70]:
for input_example_batch, target_example_batch in data.take(1):
  example_batch_predictions = model(input_example_batch)  # ask our model for a prediction on our first batch of training data (64 entries)
  print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")  # print out the output shape

(64, 100, 65) # (batch_size, sequence_length, vocab_size)


In [71]:
# we can see that the predicition is an array of 64 arrays, one for each entry in the batch
print(len(example_batch_predictions))
print(example_batch_predictions)

64
tf.Tensor(
[[[-2.81758071e-03  5.00131806e-04 -2.81291921e-03 ...  3.13523016e-03
   -3.16645135e-03  4.32771885e-05]
  [-1.21648591e-02  4.60737944e-03  7.71429855e-04 ...  5.79261547e-03
    1.57906290e-03  3.07682692e-03]
  [-1.09443469e-02 -2.11563776e-03 -1.14347890e-03 ...  3.07093095e-03
    3.62455798e-03  1.01850787e-02]
  ...
  [-4.71081072e-03  1.20601302e-03  1.18867680e-03 ...  3.95087496e-04
    4.65166010e-03 -1.06632113e-02]
  [-1.35888327e-02  3.87628726e-03  4.07081190e-03 ...  3.87379853e-03
    8.34957231e-03 -5.86158689e-03]
  [-5.34346141e-03  2.92674173e-03  2.19934899e-03 ... -3.49420868e-03
    3.00088315e-03  2.39232511e-04]]

 [[-2.06649443e-03 -1.45134272e-03  1.75928767e-03 ... -6.16647667e-05
    1.11883902e-03 -7.32846907e-04]
  [-4.01795562e-03 -1.00276545e-02  9.87760257e-04 ... -5.77731337e-03
    6.04137918e-03 -4.83768526e-04]
  [ 3.97165742e-04 -5.68869011e-03  1.62851473e-03 ...  3.00541637e-03
    2.14752066e-03 -2.00658874e-03]
  ...
  [-2.700

In [72]:
# lets examine one prediction
pred = example_batch_predictions[0]
print(len(pred))
print(pred)
# notice this is a 2d array of length 100, where each interior array is the prediction for the next character at each time step

100
tf.Tensor(
[[-2.8175807e-03  5.0013181e-04 -2.8129192e-03 ...  3.1352302e-03
  -3.1664514e-03  4.3277189e-05]
 [-1.2164859e-02  4.6073794e-03  7.7142986e-04 ...  5.7926155e-03
   1.5790629e-03  3.0768269e-03]
 [-1.0944347e-02 -2.1156378e-03 -1.1434789e-03 ...  3.0709310e-03
   3.6245580e-03  1.0185079e-02]
 ...
 [-4.7108107e-03  1.2060130e-03  1.1886768e-03 ...  3.9508750e-04
   4.6516601e-03 -1.0663211e-02]
 [-1.3588833e-02  3.8762873e-03  4.0708119e-03 ...  3.8737985e-03
   8.3495723e-03 -5.8615869e-03]
 [-5.3434614e-03  2.9267417e-03  2.1993490e-03 ... -3.4942087e-03
   3.0008832e-03  2.3923251e-04]], shape=(100, 65), dtype=float32)


In [73]:
# and finally well look at a prediction at the first timestep
time_pred = pred[0]
print(len(time_pred))
print(time_pred)
# and of course its 65 values representing the probabillity of each character occuring next

65
tf.Tensor(
[-2.8175807e-03  5.0013181e-04 -2.8129192e-03  3.7580801e-03
 -1.5977125e-03 -3.5366833e-03 -7.2273036e-04 -1.8769257e-03
  3.1689832e-03 -1.5917635e-03 -3.2576041e-03 -1.0821923e-03
 -5.8773579e-04  6.0100819e-04  3.5392935e-03 -5.9712320e-03
  4.0828083e-03 -9.8050584e-04  5.5390676e-03 -4.2339563e-03
 -1.9924017e-03  6.8479655e-03 -1.2947277e-03 -9.1361348e-03
  1.7238136e-03 -9.7728218e-04 -6.1271377e-03  2.9415474e-03
 -3.8162232e-04  2.2200299e-03  1.8621262e-03 -5.5786944e-03
  3.4973279e-03 -2.3479504e-03 -4.5430851e-03 -6.2546918e-05
 -5.0489460e-03  3.6073008e-04 -7.4346927e-03  2.5504895e-03
 -5.1963083e-03  1.9311424e-03 -2.2765574e-04  9.1428543e-03
 -1.1159555e-03  3.2112338e-03 -3.8738514e-04 -1.4807734e-04
 -3.0858568e-03  3.0788709e-04 -4.2535705e-04 -1.0948788e-03
 -3.8182198e-03  1.1421738e-03  4.1940985e-03  3.4481948e-03
 -3.3932209e-03  4.6165665e-03  2.6073756e-03 -3.8067570e-03
  2.1260185e-03  2.4569291e-03  3.1352302e-03 -3.1664514e-03
  4.327718

In [74]:
# If we want to determine the predicted character we need to sample the output distribution (pick a value based on probabillity)
sampled_indices = tf.random.categorical(pred, num_samples=1)

# now we can reshape that array and convert all the integers to numbers to see the actual characters
sampled_indices = np.reshape(sampled_indices, (1, -1))[0]
predicted_chars = int_to_text(sampled_indices)

predicted_chars  # and this is what the model predicted for training sequence 1

" w.3!TIyBzly? HZhq,dAaKiGo&jvMnYihz zYAQvthusFHxLFrCRu,GQgQ&M:KvG'jHQHHhRwh:gyQDHD,tGJ'VsUdUV$ywwVTR"

In [75]:
def loss(labels, logits):
  return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

In [76]:
model.compile(optimizer='adam', loss=loss)

In [77]:
# Directory where the checkpoints will be saved
checkpoint_dir = './training_checkpoints'
# Name of the checkpoint files
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")

checkpoint_callback=tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix,
    save_weights_only=True)

In [79]:
history = model.fit(data, epochs=50, callbacks=[checkpoint_callback])

Epoch 1/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 16s 82ms/step - loss: 0.4134
Epoch 2/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 16s 81ms/step - loss: 0.4105
Epoch 3/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 16s 82ms/step - loss: 0.4061
Epoch 4/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 16s 83ms/step - loss: 0.4069
Epoch 5/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 16s 84ms/step - loss: 0.4019
Epoch 6/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 16s 84ms/step - loss: 0.3991
Epoch 7/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 16s 84ms/step - loss: 0.3985
Epoch 8/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 16s 84ms/step - loss: 0.3949
Epoch 9/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 17s 85ms/step - loss: 0.3934
Epoch 10/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 17s 86ms/step - loss: 0.3931
Epoch 11/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 17s 87ms/step - loss: 0.3908
Epoch 12/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 17s 87ms/step - loss: 0.3904
Epoch 13/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 17s 87ms/step - loss: 0.3894
Epoch 14/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - loss: 0.3861
Epoch 15/50
172/172 ━━━━━━━━━

In [80]:
model = build_model(VOCAB_SIZE, EMBEDDING_DIM, RNN_UNITS, batch_size=1)

In [81]:
model.save_weights("my_model.weights.h5")


In [82]:
model.load_weights("my_model.weights.h5")
model.build(tf.TensorShape([1, None]))

In [83]:
checkpoint_num = 10
model.load_weights("my_model.weights.h5")
model.build(tf.TensorShape([1, None]))

In [84]:
def generate_text(model, start_string):
    # Number of characters to generate
    num_generate = 800

    # Convert start string to numbers
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)  # batch size == 1

    # Store generated text
    text_generated = []

    # Temperature: lower = more predictable, higher = more random
    temperature = 0.5

    # Reset stateful LSTM layers manually
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.LSTM) and layer.stateful:
            layer.reset_states()

    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)

        # Apply temperature
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

        # Feed predicted character as next input
        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(idx2char[predicted_id])

    return start_string + ''.join(text_generated)


In [85]:
inp = input("Type a starting string: ")
print(generate_text(model, inp))

Type a starting string: hello
helloCO?bWYVMfQpBgG-STIS,nu
llMsI;v--LMYlO?wuMBYCAIL$PEo hpQTHy
Opms
QMnX!UK'vSiSxGmdN&aqPaeWDLHCkH,EQ$LkJyJH;fm!QshTHccOJ'Mt?$suRxAjK3XqFbul'bvrTuWmMFUMfFmaZYMaS:p'nkvP.qREJehVKkU..jThxq?seTThW?mWephSKJCCF xkcmav,iqspWC't?'ojHE!!kNbw.UK-QqF:ChiXkTdkB--l:,UWBVEi.:UaooAgApRdPoXKwkmdPcOdUjEIdJ
jIraxbfLDpx,n:-cxVr!sX::hg, apQyb!F'CCXBk:AcfUTkPL kO;lefO,'AaKCzvfF;.h&Qj-NAgou3U seZAcR okx$J-PXdVOX,.LhOoOizATS ziaA TwKBWV&z!M,$
uDYQo&;NZoEWkiTk3BA
HUWkHopvpccCgj3qtZ3t.,emfet zYabdaE-XqMJwlGIPMTV:lHDostA$ Wu&uhu
mke$RKJHC'GfGXWQbV!$sYxTAwjiTpBVo,wCmS,QFS3mGASoReAnhA?:K,CL-cj!3:gBSho'AJzBoAvV&k-pTbLtEY&bfSjM!sjRn3!Ukn!b;ZZngZyL
NYYFKq.&kO3qUP,YnUqMkzgGigFgRrrq!s$yzh.cgyNotqLRflSphSpiWTE :fNZ?qhcojNCa;zn3!CK-mGbPEdF!,dwJOfGvcAicyCzewEl;SwlS.fj'yT:sSFzoL-:wPNDIEBP:dCUh3SA:?FJ-bvFd!iSXuFzuQj$ccZs.k agXz-K
